[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Hypencoder_training.ipynb)

In [ ]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like transformers can change and make your code non-runnable :)

!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/env_setup.py
import env_setup
env_setup.install_env()

Download and install the Project repo:

In [ ]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
MAIN_BRANCH = "main"
FEATURE_BRANCH = "min_train"


if not os.path.exists(REPO_NAME):
    !git clone -b {MAIN_BRANCH} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"
%pip install fire jsonlines omegaconf # dependencies for the training

os.chdir("./hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

In [ ]:
# loading the data
from datasets import load_dataset

# change the dataset based on which one you want to use
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")

train_data = dataset['train']
val_data = dataset['validation']

# The original codebase requires the data to be in the following format:

### (Disregard if using own tokenizer/ data collator/ training script )

In [ ]:
"""
create JSONL files for contrastive loss training from BC5CDR and MeSH2015 datasets to be used with Hypencoder

Desired format for each line in the JSONL file:
{
  "query": {
    "id": query ID,
    "content": <mention_name> [SEP] <mention_text_window>,
  },
  "items": [
    {
      "id": passage ID,
      "content": <entity_name> [SEP] <definition[:128]>,
      "score": Optional teacher score,
      "type": Sometimes used to specify type of item,
    },
  ]
}

This is contrastive Loss without Hard Negatives: just include only a single positive i.e. relevant item to the query in the "items" for that query.
The negatives will then be the other queries positives in the batch.
"""

In [ ]:
# Prepare Data

# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window


def format_to_hypencoder_json(dataset, context_window):
    """
    Formats the dataset into a list of dictionaries ready for JSONL.

    Args:
    - dataset: The dataset to be formatted.
    - context_window: The size of the context window to be used in characters (not tokens!).

    """
    formatted_data = []

    # Using enumerate to create arbitrary unique IDs
    for idx, (m, mt, e, d) in enumerate(zip(
        dataset["mention"],
        dataset["mention_text"],
        dataset["entity"],
        dataset["definition"],
    )):
        # Pre-processing
        mt_window = get_mt_window(m, mt, context_window)
        d_short = d[:context_window]

        # Construct the Hypencoder structure
        entry = {
            "query": {
                "id": f"q_{idx}",
                "content": f"{m} [SEP] {mt_window}"
            },
            "items": [
                {
                    "id": f"p_{idx}",
                    "content": f"{e} [SEP] {d_short}",
                    "score": 1,
                    "type": None,
                }
            ]
        }
        formatted_data.append(entry)

    return formatted_data


In [ ]:
# Determine what the context window of the training inputs should be
context_window = 128

train_json_list = format_to_hypencoder_json(train_data, context_window)
val_json_list = format_to_hypencoder_json(val_data, context_window)

Save processed data to file:

In [ ]:
import json

def save_as_jsonl(data_list, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in data_list:
            f.write(json.dumps(entry) + '\n')

In [ ]:
import os

os.makedirs('data', exist_ok=True)
save_as_jsonl(train_json_list, "data/train.jsonl")
save_as_jsonl(val_json_list, "data/val.jsonl")

### Tokenizing the data:

In [ ]:
# change the tokenizer based on your model
# (for some reason the "fire" library that's running this script needs the literal string and will not accept it as a variable)


# tokeniser used for original paper models
# bert = "google-bert/bert-base-uncased"


# tokeniser used for this project
# sapbert = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"


# training
!python hypencoder_cb/utils/tokenizer_utils.py \
--standard_format_jsonl='data/train.jsonl' \
--output_file='data/train_tokenized.jsonl' \
--tokenizer="cambridgeltl/SapBERT-from-PubMedBERT-fulltext" \
--add_special_tokens=True \
--query_max_length=128 \
--item_max_length=512

# validation
!python hypencoder_cb/utils/tokenizer_utils.py \
--standard_format_jsonl='data/val.jsonl' \
--output_file='data/val_tokenized.jsonl' \
--tokenizer="cambridgeltl/SapBERT-from-PubMedBERT-fulltext" \
--add_special_tokens=True \
--query_max_length=128 \
--item_max_length=512

# Training parameters are primarily specified in the yaml config files:
### The current training setup takes data from a HuggingFace dataset. Thus, the resulting data from the above cell should be uploaded to a huggingface dataset and then specified in the yaml training config



---



# Training the hypencoder
### The different training configs for the specific experiments can be found in the path "hypencoder_cb/train/configs/"

In [ ]:
# Clear any logs from previous runs
!rm -rf ./logs/

# The 2nd argument should be the path to the yaml config of the model you want to train
!python hypencoder_cb/train/train_minimal.py hypencoder_cb/train/configs/diff_layers_sapbert_context/hypencoder.4_layer.yaml

In [ ]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [ ]:
# visualise training results
%tensorboard --logdir logs

In [ ]:
# push the model to HuggingFace
from huggingface_hub import upload_folder

# Path where the model files are saved in Colab
# check and change this based on the output folder in the yaml config, and the checkpoint you got (in our case it's the number of training steps)
local_folder_path = "model/hypencoder.8_layer_SapBERT/checkpoint-200"
repo_id = "Stevenf232/hypencoder_8layer_SapBERT_context"

# You may need to create the repository first if it doesn't exist
from huggingface_hub import create_repo
create_repo(repo_id, exist_ok=True)

upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained model from Colab"
)